# 수정본의 실행·해석 범위

이 노트북은 2026-10-07 코드 수정본입니다. 저장된 출력과 과거 성능 숫자는 제거했습니다. 실제 Excel 전체 처리, 모델 학습·탐색과 성능 재평가는 이번 수정에서 실행하지 않았습니다. 노트북 폴더를 작업 디렉터리로 사용하고 원자료와 `ml_validation.py`, 프로젝트 의존성을 준비한 뒤 위에서 아래로 실행해야 합니다.

`수익률`은 기존 Excel 값을 신뢰하지 않고 매수가·매도가·구매주식수·투자원금으로 다시 계산합니다. 수수료·세금·배당을 포함하지 않은 계산입니다. 이전에 열람한 2021년 이후 자료는 **과거 자료 재평가**이며, 새로 봉인된 최종 테스트가 아닙니다. 전년도 재무제표라는 이유만으로 매수 당시 공개된 정보라고 보장할 수 없으므로 공시 시점 자료의 연결은 별도 과제입니다. 입력 계약과 한계는 `NOTEBOOK-REPAIR-NOTES.md`를 확인하세요.

현재 보관된 `ALL_profit_final(개).xlsx`의 13,979행 중 3행은 매도가가 없습니다(Excel 행 942, 952, 2536). 이 파일을 그대로 읽으면 `recalculate_returns`가 의도적으로 오류를 냅니다. 결측을 0으로 바꾸거나 행을 자동 삭제해 학습을 진행하지 않습니다. 원가격·보유기간 자료를 확인해 처리 근거를 정하기 전에는 전체 학습을 재개할 수 없습니다. 초기 회계 진단 비교에서 유효 13,976행의 포지션 손익 부호 141개가 양수에서 비양수로 바뀌었습니다. 이 수치는 새 가격상승 정답의 변경 개수와 다릅니다. 이는 모델 성능 검증이 아닙니다. 근거: `saved-label-audit.json`.

## 분류 목표를 가격 상승으로 분리했습니다

현재 목표는 **수정주가 기준 가격 상승 여부**입니다. `add_price_direction_targets(recalculate_returns(frame))`로 회계 열을 유지한 뒤 `가격변화율 = 100 × (매도가 / 매수가 − 1)`, `가격상승여부 = int(매도가 > 매수가)`를 추가합니다. 모델의 정답 y는 `가격상승여부`이고, `수익률_양음`은 배분된 원금·정수 주식 수에 따른 회계 손익 부호로 보존합니다.

원자료의 0주 행 25개 중 가격이 상승한 8행은 포지션 이익 0이지만 가격상승 정답은 1입니다(`verification/zero-share-audit.json`). 수정주가는 당시 체결 가능한 실제 호가와 같다고 보장되지 않습니다. 당시 가격 없이 수정주가로 정수 주식 수를 계산한 백테스트의 현실성, 원자료의 주식분할·배당 등 보정 정의는 아직 확인하지 않았습니다. 따라서 새 목표는 기업 자체의 성장·투자 수익의 증명이 아니라 저장된 수정주가의 기간 변화입니다. 기존 포지션 이익 분류의 성능 숫자를 새 목표의 성능으로 재사용하지 않습니다.

2026-10-08: 연도 키로 지표를 맞춰 평균하고 결측 지표는 선별하지 않습니다. 각 연도 회사당 500만 원으로 만든 독립 학습용 관측이며 연속 재투자 성과가 아닙니다. 매도 결과 결측은 여전히 중단합니다. Q1 가격은 두 끝 관측값 평균입니다.


In [ ]:
import os

import value_factor_backtest as kse_functions
import pandas as pd
import numpy as np
from ml_validation import recalculate_returns, add_price_direction_targets
from database.stock_etl import build_adjusted_closes

### 수정주가 데이터프레임

In [ ]:
# 엑셀 파일 경로 (파일 경로는 실제 경로로 수정해야 합니다)
file_path = 'KSE_FIN_DATA_2023.xlsx'

if 'MYSQL_HOST' in os.environ:
    import sqlalchemy

    engine = sqlalchemy.create_engine(
        sqlalchemy.URL.create(
            'mysql+pymysql',
            username=os.environ['MYSQL_USER'],
            password=os.environ['MYSQL_PASSWORD'],
            host=os.environ['MYSQL_HOST'],
            database=os.environ.get('MYSQL_DATABASE', 'stock_data'),
        )
    )
    df_close = build_adjusted_closes(engine)
else:
    df_close = pd.read_excel(file_path, sheet_name='수정주가')

df_close['Symbol Name'] = pd.to_datetime(df_close['Symbol Name'])

### 1월 첫 관측과 3월 마지막 관측 수정주가의 평균(일별 평균 아님)

In [ ]:
company_columns = [col for col in df_close.columns if col != 'Symbol Name']

average_prices_df = pd.DataFrame()

for year in range(2000, 2024):  # 연도 범위는 실제 데이터에 맞게 조정하세요
    q1_averages = kse_functions.calculate_q1_endpoint_mean(
        df_close, year, company_columns
    )
    q1_averages['Year'] = year
    average_prices_df = pd.concat(
        [average_prices_df, pd.DataFrame([q1_averages])], ignore_index=True
    )

average_prices_df = average_prices_df[
    ['Year'] + [col for col in average_prices_df.columns if col != 'Year']
]

average_prices_df['Year'] = average_prices_df['Year'].astype(int)

average_prices_df

### PER·PBR·PSR·PCR

### 재무 데이터

In [ ]:
df_fin = pd.read_excel(file_path)

In [ ]:
base_sale_month = 3
base_sale_year = 2002
base_purchase_month = 4
base_purchase_year = 2001

start_year = 2001
end_year = 2022

연도는 앞선 반복문에서 붙입니다. `recalculate_returns`가 회계 손익·수익률·수익률_양음을 계산하고, `add_price_direction_targets`가 별도 가격변화율·가격상승여부를 추가합니다. Excel에서 정답을 수동 작성하지 않습니다.

In [ ]:
PER_SHARE_COLUMNS = {'PER': 'EPS', 'PBR': 'BPS', 'PSR': 'SPS', 'PCR': 'CFPS'}


def build_indicator_tables(ratio, column):
    per_share_values = kse_functions.extract_per_share_values(df_fin, column)
    price_ratios = kse_functions.calculate_price_ratios(
        per_share_values, column, average_prices_df, company_columns
    )
    inverse_ratios = kse_functions.calculate_inverse_price_ratios(
        price_ratios, company_columns
    )
    ranked_companies_by_year = kse_functions.calculate_price_ratio_rank(
        inverse_ratios, company_columns
    )
    combined_profit = kse_functions.calculate_combined_profit(
        ranked_companies_by_year,
        start_year,
        end_year,
        base_purchase_year,
        base_purchase_month,
        base_sale_year,
        base_sale_month,
        df_close,
    )
    df_profit = add_price_direction_targets(
        recalculate_returns(combined_profit.reset_index(drop=True))
    )
    df_profit_cleaned = kse_functions.merge_values(
        df_profit, inverse_ratios, f'1/{ratio}'
    )
    df_profit_cleaned.dropna(subset=[f'1/{ratio}'], inplace=True)
    df_profit_cleaned[f'1/{ratio} Rank'] = df_profit_cleaned.groupby('연도')[
        f'1/{ratio}'
    ].rank(ascending=False)
    return inverse_ratios, df_profit_cleaned


inverse_ratios_by_indicator = {}
for ratio, column in PER_SHARE_COLUMNS.items():
    inverse_ratios_by_indicator[ratio], df_profit_cleaned = (
        build_indicator_tables(ratio, column)
    )
    output_filename = f'{ratio}_profit_final.xlsx'
    df_profit_cleaned.to_excel(output_filename, index=False)
    print(f'Data saved to {output_filename}')

### 동일비중 전략

In [ ]:
data_frames = list(inverse_ratios_by_indicator.values())

df_inverse_all = kse_functions.calculate_weighted_average(data_frames)

top_all_year = kse_functions.calculate_price_ratio_rank(
    df_inverse_all, company_columns
)

combined_profit = kse_functions.calculate_combined_profit(
    top_all_year,
    start_year,
    end_year,
    base_purchase_year,
    base_purchase_month,
    base_sale_year,
    base_sale_month,
    df_close,
)
print(combined_profit)

In [ ]:
df_all_profit = add_price_direction_targets(
    recalculate_returns(combined_profit.reset_index(drop=True))
)

df_merged = df_all_profit
for ratio, inverse_ratios in inverse_ratios_by_indicator.items():
    df_merged = kse_functions.merge_values(
        df_merged, inverse_ratios, f'1/{ratio}'
    )
df_merged = kse_functions.merge_values(df_merged, df_inverse_all, 'ALL')

df_all_profit_cleaned = df_merged.copy()

df_all_profit_cleaned.dropna(subset=['ALL'], inplace=True)

df_all_profit_cleaned['ALL Rank'] = df_all_profit_cleaned.groupby('연도')[
    'ALL'
].rank(ascending=False)

output_filename = 'ALL_profit_final.xlsx'
df_all_profit_cleaned.to_excel(output_filename, index=False)

print(f'Data saved to {output_filename}')

### 네가지 지표를 모두 활용하는 경우 지표별 순위를 기준으로 측정

In [ ]:
for ratio in PER_SHARE_COLUMNS:
    df_all_profit_cleaned[f'1/{ratio} Rank'] = df_all_profit_cleaned.groupby(
        '연도'
    )[f'1/{ratio}'].rank(ascending=False)
df_all_profit_cleaned['ALL Rank'] = df_all_profit_cleaned.groupby('연도')[
    'ALL'
].rank(ascending=False)

df_all_profit_cleaned

In [ ]:
output_filename = 'ALL_profit_final(개).xlsx'
df_all_profit_cleaned.to_excel(output_filename, index=False)

print(f'Data saved to {output_filename}')